# Python机器学习基础教程 ch08：全书总结

对应笔记 [ch08.md](../ch08.md)。本 notebook 是**分节跟随代码**：小节结构与笔记一致，完整讲解、原书插图与“坑与陷阱”请看笔记页面。

运行环境：`machinelearn`（Python 3.10 / scikit-learn 1.7 / pandas 2.3 / mglearn，mglearn 已打 boston 本地数据补丁）。原书 0.19 时代 API 的适配处在代码注释中以 **版本适配** 标出，与笔记中的“版本差异” admonition 一一对应。

In [1]:
# ===== 公共设置（notebook 版新增；原书在 Jupyter 全局环境中运行）=====
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import mglearn

# 兜底：nbclient/nbconvert 实跑时 inline 后端的 flush_figures 钩子未注册，
# 图会留在内存不进输出——这里手动补一个“每个 cell 结束时把打开的图显示出来”
import IPython
_ip = IPython.get_ipython()
if _ip is not None:
    def _flush_figs():
        if plt.get_fignums():
            plt.show()
    _ip.events.register("post_execute", _flush_figs)

print("sklearn", sklearn.__version__, "| pandas", pd.__version__,
      "| numpy", np.__version__)

sklearn 1.7.0 | pandas 2.3.3 | numpy 2.2.6


**本章解决**：全书最后一章不教新算法，教"怎么把前 7 章真正用起来"。动手之前怎么拆解问题、怎么定义并衡量成功（8.1）；原型怎么走向生产环境（8.2）；上线之后怎么在线测试、做 A/B 测试（8.3）；当 scikit-learn 没有你要的处理方法时，怎么按它的接口规矩写自己的估计器并接进 Pipeline 与 GridSearchCV（8.4，直接呼应第 6 章的管道与交叉验证）；最后给出继续深造的书单与路线图（8.5）。8.6 一句话收尾。

## 8.1 处理机器学习问题

学完全书这些强大的方法，你可能很想马上拿最喜欢的算法去解决一个数据问题。作者明确说：**这通常不是开始分析的好方法**。机器学习算法通常只是更大的数据分析与决策过程的一小部分。为了有效利用机器学习，要退后一步，全面地思考问题。

## 8.2 从原型到生产

本书讨论的工具对许多机器学习应用都很好用，可以非常快速地进行分析和原型设计。许多组织——甚至是非常大型的组织，比如国际银行和全球社交媒体公司——也将 Python 和 scikit-learn 用于生产系统。但许多公司拥有复杂的基础架构，把 Python 集成进去并不总是容易。这不一定是个问题：常见的分工是数据分析团队和生产团队各用各的语言。

## 8.3 测试生产系统

本书介绍的评估都基于**事先收集的测试集**，这叫**离线评估（offline evaluation）**。但如果你的机器学习系统是面向用户的，这只是评估算法的第一步。下一步通常是在线测试（online testing）或实时测试（live testing）：对算法在**整个系统中**使用的结果进行评估。

## 8.4 构建你自己的估计器

本书包含 scikit-learn 中实现的大量工具和算法，可用于各种类型的任务。但你通常需要对数据做一些**特殊处理**，这些处理方法没有在 scikit-learn 中实现。分两种情况：

In [2]:
# ===== 书 8.4 节 =====
from sklearn.base import BaseEstimator, TransformerMixin

class MyTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, first_parameter=1, second_parameter=2):
        # 所有参数必须在__init__函数中指定
        self.first_parameter = 1        # ← 原书笔误，应为 self.first_parameter = first_parameter
        self.second_parameter = 2       # ← 原书笔误，应为 self.second_parameter = second_parameter

    def fit(self, X, y=None):
        # fit应该只接受X和y作为参数
        # 即使你的模型是无监督的，你也需要接受一个y参数！

        # 下面是模型拟合的代码
        print("fitting the model right here")
        # fit返回self
        return self

    def transform(self, X):
        # transform只接受X作为参数

        # 对X应用某种变换
        X_transformed = X + 1
        return X_transformed

In [3]:
# ===== 书 8.4 节 =====
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# 自定义估计器：把第 feature 列替换成"是否大于 threshold"的布尔特征
class ThresholdEstimator(BaseEstimator, TransformerMixin):
    def __init__(self, feature=0, threshold=0.0):
        # 所有参数必须在 __init__ 中保存为同名属性（8.4 节的规矩）
        self.feature = feature
        self.threshold = threshold

    def fit(self, X, y=None):
        # fit 只接受 X 和 y；这里没有需要从数据里学的量，什么都不用记
        return self                            # 必须返回 self

    def transform(self, X):
        # transform 只接受 X
        X_transformed = X.copy()               # 不改动原始输入
        # 用布尔特征替换原列：True/False 写进浮点数组时变成 1.0/0.0
        X_transformed[:, self.feature] = X[:, self.feature] > self.threshold
        return X_transformed

# 接入 Pipeline + GridSearchCV：依赖数据的预处理从此与交叉验证同处一个循环
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=0)

pipe = make_pipeline(
    ThresholdEstimator(),          # 自定义变换器，与其他 sklearn 组件完全同构
    StandardScaler(),
    LogisticRegression(max_iter=5000))

# 双下划线寻址（第 6 章）：make_pipeline 的步骤名是类名小写 thresholdestimator
param_grid = {
    'thresholdestimator__feature':   [7, 27],                     # mean / worst concave points
    'thresholdestimator__threshold': [0.0, 0.02, 0.05, 0.08, 0.1],
}
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)
print(grid.best_params_)
print("交叉验证最优分数: {:.3f}".format(grid.best_score_))
print("测试集分数: {:.3f}".format(grid.score(X_test, y_test)))

{'thresholdestimator__feature': 7, 'thresholdestimator__threshold': 0.05}
交叉验证最优分数: 0.979
测试集分数: 0.965


## 8.5 下一步怎么走

本书对机器学习做了介绍，目标是让你成为一名高效的从业者。如果你想进一步提高技能，下面是关于书籍和更专业资源的建议——按方向分七路。

### 8.5.1 理论

本书试图直观地解释常见机器学习算法的工作原理，不要求你在数学或计算机科学方面有坚实基础。但讨论的许多模型都用了概率论、线性代数和最优化方面的理论。理解算法的所有实现细节并无必要，但了解一些理论可以让你成为更优秀的数据科学家。如果本书激起了你对机器学习的兴趣，建议**至少挑选一本**深入阅读。

### 8.5.2 其他机器学习框架和包

scikit-learn 是作者最喜欢的机器学习软件包，Python 是他们最喜欢的机器学习语言——但根据你的需求，特定情况下它们可能不是最佳选择。通常情况下，Python 很适合尝试与评估模型；更大型的 Web 服务和应用更常用 Java 或 C++ 编写，部署模型可能需要与这些系统集成。考虑 scikit-learn 之外的选择还有一个理由：如果你对统计建模和推断比对预测更感兴趣，就换 statsmodels；如果你还没有专情于 Python，就看看 R。

### 8.5.3 排序、推荐系统与其他学习类型

本书是入门书，重点介绍最常见的机器学习任务：监督学习中的分类与回归，无监督学习中的聚类和信号分解。还有许多类型的机器学习，都有重要应用。其中**两个特别重要、本书没有包含**的主题：

### 8.5.4 概率建模、推断与概率编程

大部分机器学习软件包提供预定义的模型，每种模型应用一种特定算法。但许多现实世界的问题具有**特殊的结构**，如果将这种结构正确地纳入模型，可以得到性能更好的预测。这种结构通常可以用概率论的语言来表述，并且往往来自你想要预测的情况的数学模型。

### 8.5.5 神经网络

第 2 章和第 7 章都简要涉及了神经网络的主题，但这是机器学习快速发展的领域，每周都会发布新方法和新应用。机器学习和人工智能领域的最新突破都由这些进步驱动，比如 AlphaGo 程序在围棋比赛中战胜人类冠军、语音理解的性能不断提高、接近实时的语音翻译的出现。领域进步快到"当前对最新进展的任何参考很快都会过时"——Ian Goodfellow、Yoshua Bengio 和 Aaron Courville 的新书 *Deep Learning*（MIT 出版社）对这一主题做了全面介绍。

### 8.5.6 推广到更大的数据集

本书总是假设所处理的数据可以被存储为**内存（RAM）中的一个 NumPy 数组或 SciPy 稀疏矩阵**。即使现代服务器通常有数百 GB 内存，这也是你能处理数据大小的根本限制——不是所有人都买得起这么大的机器，甚至从云端供应商租一台都负担不起。不过大多数应用中，用于构建机器学习系统的数据量相对较小，很少有机器学习数据集包含数百 GB 以上的数据；多数情况下，扩大内存或租一台云机器是可行的解决方案。但如果你需要处理 **TB 级别**的数据，或者需要节省处理大量数据的费用，有两种基本策略：

### 8.5.7 磨练你的技术

与生活中的许多事情一样，只有实践才能让你成为专家。不同的任务和不同的数据集上，特征提取、预处理、可视化和模型构建可能差异很大。你或许已经足够幸运，能够访问大量数据集和任务；如果还没想到什么任务，一个好的起点是**机器学习竞赛**——发布一个数据集和一个给定任务，许多团队为得到最佳预测结果而展开竞争，公司、非盈利组织和大学都会举办这种比赛。

## 8.6 总结

全书最后只有两句话：作者希望已经让你相信了机器学习在大量应用中的实用性，以及在实践中实现机器学习的简单性。**继续挖掘数据，同时不要忽视大局。**前半句是 7 章算法与 API 的落点；后半句回到 8.1 节的方法论——算法只是数据分析与决策过程的一小部分，问题拆解、成功度量、数据本身往往比模型更重要。